# PPO VM Allocation Experiments

This notebook trains and evaluates PPO agents for VM allocation using the existing DRL pipeline:

- `rl/environment.py`: `VMAllocationEnv` (Gymnasium environment)
- `rl/config.py`: PPO and reward configuration
- `train_ppo.py`: training script
- `eval_ppo.py`: evaluation and comparison with LP baseline

You can use this notebook to:
- Run quick training experiments (e.g., smaller `total_timesteps`)
- Evaluate PPO against the LP baseline
- Inspect the generated PPO schedules and metrics.


In [1]:
# Imports and configuration

from pathlib import Path

# Module imports
import importlib
import train_ppo as train_ppo_module
import eval_ppo as eval_ppo_module
import rl.config as rl_config_module

from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST
from train_ppo import train_ppo
from eval_ppo import evaluate_scenario, print_comparison

# Reload modules to pick up latest code when notebook stays open
importlib.reload(train_ppo_module)
importlib.reload(eval_ppo_module)
importlib.reload(rl_config_module)

# Refresh config after reload
from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

# Show current default PPO configuration
config = PPOConfig()
config


Project root: e:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure


PPOConfig(learning_rate=0.0003, n_steps=2048, batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, clip_range_vf=None, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5, episode_length=480, horizon=18, total_timesteps=1000000, tensorboard_log='./tensorboard_logs/', log_interval=10, save_freq=10000)

In [2]:
# Quick training config (adjust as needed)
from copy import deepcopy

# Number of parallel envs (increase if you have CPU/GPU resources)
N_ENVS = 8

exp_config = deepcopy(config)

# Default: 1,000,000 timesteps (recommended after reward tweaks)
# For a quick debug run, set to 50_000
exp_config.total_timesteps = 1_000_000

print("Training configuration override:")
print("  total_timesteps =", exp_config.total_timesteps)
print("  episode_length  =", exp_config.episode_length)
print("  horizon         =", exp_config.horizon)
print("  n_envs          =", N_ENVS)

Training configuration override:
  total_timesteps = 1000000
  episode_length  = 480
  horizon         = 18
  n_envs          = 8


In [4]:

print("Training configuration:")
print("  total_timesteps =", exp_config.total_timesteps)
print("  episode_length  =", exp_config.episode_length)
print("  horizon         =", exp_config.horizon)
print("  n_envs          =", N_ENVS)

# Train overload-first scenario
model_overload = train_ppo(
    scenario=SCENARIO_OVERLOAD,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)

# Train cost-first scenario
model_cost = train_ppo(
    scenario=SCENARIO_COST,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)


Training configuration:
  total_timesteps = 1000000
  episode_length  = 480
  horizon         = 18
  n_envs          = 8

Training PPO for scenario: OVERLOAD
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_forest
  Target: cpu_total_usage
  Saved at: 2025-11-11 22:19:30
Latest model: models\svr_memory_usage_pct_20251111_184735.pkl
✓ Model loaded: models\svr_memory_usage_pct_20251111_184735.pkl
  Model: svr
  Target: memory_usage_pct
  Saved at: 2025-11-11 18:47:35
Latest model: models\random_forest_system_load_20251111_221944.pkl
✓ Model loaded: models\random_forest_system_load_20251111_221944.pkl
  Model: random_forest
  Target: system_load
  Saved at: 2025-11-11 22:19:44
Creating new PPO model
Using cpu device

Starting training for 1,000,000 timesteps...
Logging to ./tensorboard_logs/PPO_9


Output()

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 480       |
|    ep_rew_mean     | -1.05e+04 |
| time/              |           |
|    fps             | 733       |
|    iterations      | 1         |
|    time_elapsed    | 22        |
|    total_timesteps | 16384     |
----------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 480         |
|    ep_rew_mean          | -1.05e+04   |
| time/                   |             |
|    fps                  | 597         |
|    iterations           | 2           |
|    time_elapsed         | 54          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.017610408 |
|    clip_fraction        | 0.191       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.58       |
|    explained_variance   | -0.691      |
|    learning_rate        | 0.0003      |
|    loss                 | 0.272       |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0294     |
|    value_loss           | 0.186       |
-----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 480         |
|    ep_rew_mean          | -1.04e+04   |
| time/                   |             |
|    fps                  | 571         |
|    iterations           | 3           |
|    time_elapsed         | 85          |
|    total_timesteps      | 49152       |
| train/                  |             |
|    approx_kl            | 0.017722454 |
|    clip_fraction        | 0.213       |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.56       |
|    explained_variance   | 0.264       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.131      |
|    n_updates            | 20          |
|    policy_gradient_loss | -0.0319     |
|    value_loss           | 0.0209      |
-----------------------------------------


...

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 480       |
|    ep_rew_mean     | -1.02e+03 |
| time/              |           |
|    fps             | 480       |
|    iterations      | 59        |
|    time_elapsed    | 2010      |
|    total_timesteps | 966656    |
----------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 480         |
|    ep_rew_mean          | -971        |
| time/                   |             |
|    fps                  | 480         |
|    iterations           | 60          |
|    time_elapsed         | 2044        |
|    total_timesteps      | 983040      |
| train/                  |             |
|    approx_kl            | 0.018607695 |
|    clip_fraction        | 0.207       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.01       |
|    explained_variance   | 0.972       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0711     |
|    n_updates            | 590         |
|    policy_gradient_loss | -0.0332     |
|    value_loss           | 0.000246    |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 480        |
|    ep_rew_mean          | -886       |
| time/                   |            |
|    fps                  | 481        |
|    iterations           | 61         |
|    time_elapsed         | 2076       |
|    total_timesteps      | 999424     |
| train/                  |            |
|    approx_kl            | 0.01934631 |
|    clip_fraction        | 0.212      |
|    clip_range           | 0.2        |
|    entropy_loss         | -3.95      |
|    explained_variance   | 0.97       |
|    learning_rate        | 0.0003     |
|    loss                 | -0.0434    |
|    n_updates            | 600        |
|    policy_gradient_loss | -0.0317    |
|    value_loss           | 0.000192   |
----------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 480         |
|    ep_rew_mean          | -864        |
| time/                   |             |
|    fps                  | 482         |
|    iterations           | 62          |
|    time_elapsed         | 2107        |
|    total_timesteps      | 1015808     |
| train/                  |             |
|    approx_kl            | 0.019504689 |
|    clip_fraction        | 0.221       |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.88       |
|    explained_variance   | 0.945       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0533     |
|    n_updates            | 610         |
|    policy_gradient_loss | -0.0313     |
|    value_loss           | 0.000189    |
-----------------------------------------



Training completed in 0:35:18.439536
Model saved to E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_cost.zip
VecNormalize saved to E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_cost_vecnormalize.pkl


In [3]:
# Evaluation: run PPO on test set and compare with LP baseline

results = {}

for scenario in [SCENARIO_OVERLOAD, SCENARIO_COST]:
    comp = evaluate_scenario(scenario)
    results[scenario] = comp
    print_comparison(comp)

results



Evaluating scenario: OVERLOAD
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_forest
  Target: cpu_total_usage
  Saved at: 2025-11-11 22:19:30
Latest model: models\svr_memory_usage_pct_20251111_184735.pkl
✓ Model loaded: models\svr_memory_usage_pct_20251111_184735.pkl
  Model: svr
  Target: memory_usage_pct
  Saved at: 2025-11-11 18:47:35
Latest model: models\random_forest_system_load_20251111_221944.pkl
✓ Model loaded: models\random_forest_system_load_20251111_221944.pkl
  Model: random_forest
  Target: system_load
  Saved at: 2025-11-11 22:19:44
Loaded model from E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_overload.zip

Running PPO rollout for scenario: overload
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_for

{'overload': {'scenario': 'overload',
  'timestamp': '2025-12-11 00:17:55',
  'ppo': {'total_vm_cost': 29.113599999999998,
   'total_switching_cost': 4.98,
   'total_cost': 34.093599999999995,
   'sla_violations': 4,
   'sla_violation_rate': 0.008333333333333333,
   'mean_cpu_utilization': 0.9006508297720798,
   'mean_mem_utilization': 0.0,
   'n_steps': 480},
  'ppo_30min': {'total_vm_cost': 0.0,
   'total_switching_cost': 0.0,
   'total_cost': 0.0,
   'sla_violations': 4,
   'sla_violation_rate': 0.023905723905723906,
   'n_buckets': 9}},
 'cost': {'scenario': 'cost',
  'timestamp': '2025-12-11 00:19:34',
  'ppo': {'total_vm_cost': 36.812799999999996,
   'total_switching_cost': 5.38,
   'total_cost': 42.192800000000005,
   'sla_violations': 0,
   'sla_violation_rate': 0.0,
   'mean_cpu_utilization': 0.0510493736219017,
   'mean_mem_utilization': 0.0,
   'n_steps': 480},
  'ppo_30min': {'total_vm_cost': 0.0,
   'total_switching_cost': 0.0,
   'total_cost': 0.0,
   'sla_violations': 0,

In [4]:
# Inspect generated PPO schedules and comparison JSON

import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("forecast_result")

ppo_overload_path = RESULTS_DIR / "ppo_schedule_test_overload.csv"
ppo_cost_path = RESULTS_DIR / "ppo_schedule_test_cost.csv"
comparison_path = RESULTS_DIR / "ppo_vs_lp_comparison.json"

print("PPO overload schedule exists:", ppo_overload_path.exists())
print("PPO cost schedule exists:", ppo_cost_path.exists())
print("Comparison JSON exists:", comparison_path.exists())

if ppo_overload_path.exists():
    df_over = pd.read_csv(ppo_overload_path)
    display(df_over.head())

if ppo_cost_path.exists():
    df_cost = pd.read_csv(ppo_cost_path)
    display(df_cost.head())

# Show 30-minute bucketed outputs (align with LP)
ppo_overload_bucket = RESULTS_DIR / "ppo_schedule_30min_overload.csv"
ppo_cost_bucket = RESULTS_DIR / "ppo_schedule_30min_cost.csv"
print("PPO overload 30min bucket exists:", ppo_overload_bucket.exists())
print("PPO cost 30min bucket exists:", ppo_cost_bucket.exists())

if ppo_overload_bucket.exists():
    df_over_b = pd.read_csv(ppo_overload_bucket)
    display(df_over_b.head())

if ppo_cost_bucket.exists():
    df_cost_b = pd.read_csv(ppo_cost_bucket)
    display(df_cost_b.head())

if comparison_path.exists():
    import json
    with open(comparison_path, "r") as f:
        comp_data = json.load(f)
    comp_data


PPO overload schedule exists: True
PPO cost schedule exists: True
Comparison JSON exists: False


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0,0.0,0.0,0


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0,0.0,0.0,0


PPO overload 30min bucket exists: True
PPO cost 30min bucket exists: True


,timestamp,vm_cost_per_hour_sum,switching_cost_sum,total_cost_per_hour_sum,cpu_required_cores_max,mem_required_gb_max,cpu_overflow_cores_max,mem_overflow_gb_max,cpu_vm_only_max,mem_vm_only_max,sla_violation_flag_sum,sla_violation_flag_mean
0,1970-01-25 01:00:00,0.0000,0.00,0.0000,0.670,0.319772,0.00,0,0.0,0.0,0,0.000000
1,1970-01-25 01:30:00,2.8288,1.21,4.0388,2.090,0.291744,1.39,0,26.0,136.0,1,0.016667
2,1970-01-25 02:00:00,0.0000,0.00,0.0000,0.680,0.281257,0.00,0,0.0,0.0,0,0.000000
3,1970-01-25 02:30:00,0.0000,0.00,0.0000,0.467,0.277264,0.00,0,0.0,0.0,0,0.000000
4,1970-01-25 03:00:00,2.0480,0.59,2.6380,1.660,0.279743,0.96,0,36.0,156.0,0,0.000000


,timestamp,vm_cost_per_hour_sum,switching_cost_sum,total_cost_per_hour_sum,cpu_required_cores_max,mem_required_gb_max,cpu_overflow_cores_max,mem_overflow_gb_max,cpu_vm_only_max,mem_vm_only_max,sla_violation_flag_sum,sla_violation_flag_mean
0,1970-01-25 01:00:00,0.0000,0.00,0.0000,0.670,0.319772,0.00,0,0.0,0.0,0,0.0
1,1970-01-25 01:30:00,14.4544,2.02,16.4744,2.090,0.291744,1.39,0,134.0,828.0,0,0.0
2,1970-01-25 02:00:00,0.0000,0.00,0.0000,0.680,0.281257,0.00,0,0.0,0.0,0,0.0
3,1970-01-25 02:30:00,0.0000,0.00,0.0000,0.467,0.277264,0.00,0,0.0,0.0,0,0.0
4,1970-01-25 03:00:00,4.3712,1.42,5.7912,1.660,0.279743,0.96,0,80.0,312.0,0,0.0
